# NB1 — Preparación de Datos

**Plataforma:** Vast.ai (A100) — no requiere GPU.

## Propósito
Instalar dependencias, descargar y preprocesar los dos datasets utilizados en los experimentos,
y pre-descargar los modelos base para evitar re-descargas durante el entrenamiento.

## Datasets
- **MR (Movie Reviews):** clasificación binaria de sentimiento a nivel de oración.
- **SemEval 2018 Task 1 Emotions:** clasificación multi-etiqueta de 11 emociones en tweets.

## Salidas
```
/workspace/negative_supervision/
├── data/
│   ├── label_info.json
│   ├── mr/train.csv, val.csv, test.csv
│   └── semeval/train.csv, val.csv, test.csv
└── hf_cache/   <- BERT y RoBERTa descargados
```
**Ejecutar una sola vez antes de cualquier otro notebook.**

## 1. Rutas

In [ ]:
import os
import sys
from pathlib import Path

BASE_DIR    = Path('/workspace/negative_supervision')
DATA_DIR    = BASE_DIR / 'data'
MR_DIR      = DATA_DIR / 'mr'
SEMEVAL_DIR = DATA_DIR / 'semeval'

for d in [MR_DIR, SEMEVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Base: {BASE_DIR}')

## 2. Instalación de dependencias

In [ ]:
# Instalamos en el Python activo del kernel para evitar discrepancias de entorno en Vast.ai
!{sys.executable} -m pip install -q transformers==4.40.0 datasets==2.19.0 scikit-learn==1.4.2 pandas numpy sentencepiece protobuf

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Importaciones OK.')

## 3. Pre-descarga de modelos base

BERT-base y RoBERTa-base se descargan una sola vez al directorio de caché local.
Los notebooks posteriores cargarán desde este caché sin re-descargas,
lo que evita demoras significativas al inicio de cada trial de entrenamiento.

In [ ]:
os.environ['HF_HOME'] = '/workspace/hf_cache'
from transformers import AutoTokenizer, AutoModel

for model_name in ['bert-base-uncased', 'roberta-base']:
    print(f'Descargando {model_name}...')
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f'  cacheado.')

print('Modelos listos.')

## 4. Dataset MR (Movie Reviews)

Clasificación binaria de sentimiento (positivo/negativo) a nivel de oración.
Cargado desde HuggingFace `rotten_tomatoes`, que corresponde a las mismas
particiones del benchmark SentEval utilizadas en el paper original.

In [ ]:
print('Cargando MR...')
mr_raw   = load_dataset('rotten_tomatoes')
mr_train = mr_raw['train'].to_pandas()
mr_val   = mr_raw['validation'].to_pandas()
mr_test  = mr_raw['test'].to_pandas()

print(f'train: {len(mr_train):,} | val: {len(mr_val):,} | test: {len(mr_test):,}')
print(mr_train['label'].value_counts().to_string())

mr_train.to_csv(MR_DIR / 'train.csv', index=False)
mr_val.to_csv(MR_DIR   / 'val.csv',   index=False)
mr_test.to_csv(MR_DIR  / 'test.csv',  index=False)

mr_label_info = {
    'task_type':   'single_label',
    'num_labels':  2,
    'label_names': ['negativo', 'positivo'],
    'metric':      'accuracy'
}
print('MR guardado.')

## 5. Dataset SemEval 2018 Task 1 Emotions

Clasificación multi-etiqueta de emociones en tweets. 11 etiquetas con alto solapamiento
semántico (joy/optimism, fear/sadness, anger/disgust), lo cual motiva directamente el uso
de supervisión negativa: el encoder debe distinguir textos similares con etiquetas distintas.

SemEval no provee partición de validación; se reserva el 15% del train con semilla fija.

In [ ]:
print('Cargando SemEval 2018 Task 1...')
semeval_raw = load_dataset('sem_eval_2018_task_1', 'subtask5.english')

SEMEVAL_LABELS = [
    'anger', 'anticipation', 'disgust', 'fear', 'joy',
    'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'
]
NUM_SEMEVAL_LABELS = len(SEMEVAL_LABELS)

def semeval_to_dataframe(split):
    """Convierte partición SemEval a DataFrame con vector multi-hot como JSON string."""
    records = []
    for item in split:
        label_vec = [int(item[e]) for e in SEMEVAL_LABELS]
        records.append({'text': item['Tweet'], 'label_vec': json.dumps(label_vec)})
    return pd.DataFrame(records)

semeval_full = semeval_to_dataframe(semeval_raw['train'])
semeval_test = semeval_to_dataframe(semeval_raw['test'])

# Partición 85/15 con semilla fija
train_idx, val_idx = train_test_split(range(len(semeval_full)), test_size=0.15, random_state=SEED)
semeval_train = semeval_full.iloc[train_idx].reset_index(drop=True)
semeval_val   = semeval_full.iloc[val_idx].reset_index(drop=True)

print(f'train: {len(semeval_train):,} | val: {len(semeval_val):,} | test: {len(semeval_test):,}')

# Frecuencia de etiquetas
counts = np.array(semeval_train['label_vec'].apply(json.loads).tolist()).sum(axis=0)
print('\nFrecuencia por etiqueta (train):')
for i, label in enumerate(SEMEVAL_LABELS):
    print(f'  {label:<15} {int(counts[i]):>5}')

semeval_train.to_csv(SEMEVAL_DIR / 'train.csv', index=False)
semeval_val.to_csv(SEMEVAL_DIR   / 'val.csv',   index=False)
semeval_test.to_csv(SEMEVAL_DIR  / 'test.csv',  index=False)

semeval_label_info = {
    'task_type':   'multi_label',
    'num_labels':  NUM_SEMEVAL_LABELS,
    'label_names': SEMEVAL_LABELS,
    'metric':      'exact_match'
}
print('SemEval guardado.')

## 6. Guardar metadatos y verificar

In [ ]:
label_info = {'mr': mr_label_info, 'semeval': semeval_label_info}
with open(DATA_DIR / 'label_info.json', 'w') as f:
    json.dump(label_info, f, indent=2)

# Verificaciones de integridad
assert len(pd.read_csv(MR_DIR / 'train.csv')) == len(mr_train)
assert len(pd.read_csv(SEMEVAL_DIR / 'train.csv')) == len(semeval_train)
assert len(json.loads(pd.read_csv(SEMEVAL_DIR / 'train.csv')['label_vec'].iloc[0])) == NUM_SEMEVAL_LABELS

print('Verificaciones OK.')
print()
print('=== Resumen ===')
print(f'MR      | train: {len(mr_train):,} | val: {len(mr_val):,} | test: {len(mr_test):,} | 2 etiquetas  | single-label | métrica: accuracy')
print(f'SemEval | train: {len(semeval_train):,} | val: {len(semeval_val):,} | test: {len(semeval_test):,} | 11 etiquetas | multi-label  | métrica: exact_match')
print()
print('NB1 completo. Continuar con NB2.')